## **Dependencias**

In [8]:
!pip install -q agentpy numpy matplotlib seaborn

## **Imports y estilo**

In [9]:
import agentpy as ap
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
sns.set(context="notebook", style="whitegrid")

## **Parámetros**

In [10]:
params = {
    'steps': 600,          # duración en ticks (1 tick = 1 s)
    'green_ns': 20,        # VERDE para Norte-Sur
    'green_ew': 20,        # VERDE para Este-Oeste
    'yellow': 3,           # ÁMBAR
    'all_red': 1,          # ALL-RED (despeje)
    # Tasas Poisson de arribo (veh/s) por aproximación
    'lambda_N': 0.10, 'lambda_S': 0.10,
    'lambda_E': 0.10, 'lambda_W': 0.10,
    # Cinemática
    'v_free': 7.0,         # m/s
    'headway': 8.0,        # m separación mínima
    # Geometría (intersección centrada en 0,0)
    'L': 80.0,             # media-calzada (desde centro al extremo de dibujo)
    'w': 3.5               # ancho de carril
}

## Diccionario

In [11]:
# --- Definiciones para la rotonda ---
ENTRY_NS = {'N1','S2'}   # Entradas controladas juntas en fase NS
ENTRY_EW = {'E1','W2'}   # Entradas controladas juntas en fase EW
ALL_ENTRIES = ENTRY_NS | ENTRY_EW
ALL_EXITS = {'N2','E2','S1','W1'}

# Rutas válidas por origen de ENTRADA
ROTUNDA_ROUTES = {
    'N1': ['N2', 'W1', 'S1', 'E2'],     # U a N2, o salir a W1, S1, E2
    'E1': ['N2', 'W1', 'S1'],
    'W2': ['S1', 'E2', 'N2'],
    'S2': ['S1', 'E2', 'N2', 'W1'],
}

routes = {
    'N1': ['N2', 'W1', 'S1', 'E2'],  # entrada norte principal
    'E1': ['N2', 'W1', 'S1'],        # entrada este
    'W2': ['S1', 'E2', 'N2'],        # entrada oeste
    'S2': ['S1', 'E2', 'N2', 'W1'],  # entrada sur
    # salidas (N2, E2, S1, W1) solo reciben, no generan autos
}

## **Controlador de semáforos**

In [12]:
class FourWaySignals(ap.Agent):
    """Semáforo adaptativo: decide extender o cambiar fase según colas."""

    
    def setup(self, green_ns, green_ew, yellow, all_red):
        self.g_ns, self.g_ew = int(green_ns), int(green_ew)
        self.y, self.ar = int(yellow), int(all_red)
        self.phase = 0          # 0 = NS, 1 = EW
        self.sub = 'G'          # 'G','Y','AR'
        self.t_in = 0
        self.timeline = []      # historial de luces

        # parámetros heurísticos
        self.min_green = 10     # duración mínima del verde
        self.max_green = 40     # duración máxima del verde
        self.decision_interval = 5  # cada cuántos ticks reevaluar

    def lights(self):
        L = {d:'R' for d in ['N','S','E','W']}
        if self.phase == 0:
            L['N'] = L['S'] = self.sub
        else:
            L['E'] = L['W'] = self.sub
        return L

    @property
    def green_dirs(self):
        if self.sub != 'G': return set()
        return {'N','S'} if self.phase == 0 else {'E','W'}

    def count_queue(self, dirs):
        """Cuenta autos detenidos en las direcciones dadas."""
        return sum(1 for c in self.model.cars 
                   if c.origin in dirs and c.state == 'stop')

    def step(self):
        self.timeline.append((self.model.t, self.lights()))

        if self.sub == 'G':  
            # estamos en verde
            self.t_in += 1

            # solo reevaluamos cada X ticks
            if self.t_in % self.decision_interval == 0 and self.t_in >= self.min_green:

                if self.phase == 0:  # NS actual
                    q_ns = self.count_queue(['N','S'])
                    q_ew = self.count_queue(['E','W'])
                else:  # EW actual
                    q_ns = self.count_queue(['N','S'])
                    q_ew = self.count_queue(['E','W'])

                # regla: si mi cola > cola opuesta y aún < max_green → extender
                if self.phase == 0 and q_ns >= q_ew and self.t_in < self.max_green:
                    pass  # seguir en verde NS
                elif self.phase == 1 and q_ew >= q_ns and self.t_in < self.max_green:
                    pass  # seguir en verde EW
                else:
                    # si no, cambiamos a ámbar
                    self.sub, self.t_in = 'Y', 0

        elif self.sub == 'Y':
            if self.t_in >= self.y:
                self.sub, self.t_in = 'AR', 0
            else:
                self.t_in += 1

        elif self.sub == 'AR':
            if self.t_in >= self.ar:
                # cambio de fase
                self.phase = 1 - self.phase
                self.sub, self.t_in = 'G', 0
            else:
                self.t_in += 1


## **Agente Vehículo**

In [13]:


class Car(ap.Agent):
    """Auto en carril recto: entra desde un borde y atraviesa la intersección si tiene luz verde."""

    def setup(self, origin):
        self.origin = origin
        self.state = 'approach'
        self.v = self.model.p.v_free

        # elegir destino válido
        dests = self.model.routes[origin]
        self.dest = np.random.choice(dests)

        # asignar spawn y dirección según la calle de origen
        L, w = self.model.p.L, self.model.p.w
        off = w/2

        if origin == 'N1':
            self.pos = np.array([-off, L])   # spawn
            self.dir = np.array([0,-1])      # hacia el sur
            self.stopline = np.array([-off, +10])
        elif origin == 'E1':
            self.pos = np.array([L, +off])
            self.dir = np.array([-1,0])
            self.stopline = np.array([+10, +off])
        elif origin == 'W2':
            self.pos = np.array([-L, -off])
            self.dir = np.array([+1,0])
            self.stopline = np.array([-10, -off])
        elif origin == 'S2':
            self.pos = np.array([+off, -L])
            self.dir = np.array([0,+1])
            self.stopline = np.array([+off, -10])

        # ahora meta (goal) según destino
        if self.dest == 'N2':
            self.goal = np.array([+off, L])   # salida norte
        elif self.dest == 'E2':
            self.goal = np.array([L, -off])   # salida este
        elif self.dest == 'S1':
            self.goal = np.array([+off, -L])  # salida sur
        elif self.dest == 'W1':
            self.goal = np.array([-L, +off])  # salida oeste

    def dist_to(self, p): return np.linalg.norm(self.pos - p)

    def step(self):
        if self.state == 'done': return

        # Si llegó a la meta, termina
        if self.dist_to(self.goal) < 1.0:
            self.state = 'done'; return

        # Zona de decisión cerca de la stopline (3 m)
        near = self.dist_to(self.stopline) < 10.0

        # Reglas de luz
        L = self.model.ctrl.lights()
        if near and L[self.origin] != 'G':
            self.state = 'stop'
            return
        else:
            self.state = 'go'

        # Espacio de seguridad con el líder en el mismo carril
        vmax = self.v
        head = self.model.headway_ahead(self)
        if head is not None:
            gap = np.linalg.norm(head.pos - self.pos)
            if gap < self.model.p.headway: vmax = 0.0

        # Avanzar
        self.pos = self.pos + self.dir * vmax * 1.0  # dt=1 s


params_rotonda = {
    'steps': 600,
    'green_ns': 20,
    'green_ew': 20,
    'yellow': 3,
    'all_red': 1,
    'lambda_N': 0.15,   # N1
    'lambda_S': 0.10,   # S2
    'lambda_E': 0.12,   # E1
    'lambda_W': 0.08,   # W2
    'v_free': 7.0,
    'headway': 8.0,
    'L': 80.0,
    'w': 3.5,
    'ctrl_class': FourWaySignalsHeuristic  # usa heurística adaptativa
}

## **Modelo con arribos Poisson y listas de agentes**

In [16]:
class FourWayModel(ap.Model):

    def setup(self):
        p = self.p
        self.ctrl = FourWaySignals(self, p.green_ns, p.green_ew, p.yellow, p.all_red)
        self.cars = ap.AgentList(self, 0, Car)
        self.spawn_counts = {d:0 for d in ['N','S','E','W']}
        
        #  logs para métricas   
        self.total_delay = 0
        self.delay_counts = 0
        self.queue_lengths = {d:[] for d in ['N','S','E','W']}
        self.done_cars = 0

    def headway_ahead(self, me):
        same = [c for c in self.cars if c is not me and np.allclose(c.dir, me.dir)]
        if not same: return None
        ahead = []
        for c in same:
            v = c.pos - me.pos
            proj = np.dot(v, me.dir)
            if proj > 0:
                ahead.append((proj, c))
        if not ahead: return None
        return min(ahead, key=lambda x: x[0])[1]

    def spawn_poisson(self, origin, lam):
        k = np.random.poisson(lam)
        for _ in range(k):
            self.cars.append(Car(self, origin=origin))
            self.spawn_counts[origin]+=1

    def step(self):
        # 1) arribos
        self.spawn_poisson('N', self.p.lambda_N)
        self.spawn_poisson('S', self.p.lambda_S)
        self.spawn_poisson('E', self.p.lambda_E)
        self.spawn_poisson('W', self.p.lambda_W)

        # 2) señales
        self.ctrl.step()

        # 3) autos
        self.cars.step()

        # 4) limpieza de autos terminados
        finished = [c for c in self.cars if c.state == 'done']
        self.done_cars += len(finished)
        self.cars = ap.AgentList(self, [c for c in self.cars if c.state != 'done'], Car)

        # ---- NUEVO: métricas ----
        for d in ['N1','E1','W2','S2']:
            q_len = sum(1 for c in self.cars if c.origin == d and c.state == 'stop')
            self.queue_lengths[d].append(q_len)

        # tiempo de espera
        waiting = sum(1 for c in self.cars if c.state == 'stop')
        self.total_delay += waiting
        self.delay_counts += 1

    # ---- NUEVO: resumen de métricas ----
    def results(self):
        delay_avg = self.total_delay / max(1, self.delay_counts)
        max_queues = {d: max(self.queue_lengths[d]) if self.queue_lengths[d] else 0 
                    for d in self.queue_lengths}
        return {
            "delay_avg": delay_avg,
            "throughput": self.done_cars,
            "max_queues": max_queues
        }


    def results(self):
        delay_avg = self.total_delay / max(1, self.delay_counts)
        max_queues = {d: max(self.queue_lengths[d]) if self.queue_lengths[d] else 0 
                      for d in self.queue_lengths}
        return {
            "delay_avg": delay_avg,
            "throughput": self.done_cars,
            "max_queues": max_queues
        }
    
model_rotonda = FourWayModel(params)
    
#model_fixed = FourWayModel(params)
#for _ in range(params['steps']):
#    model_fixed.step()
#print("Plan fijo:", model_fixed.results())

#model_heur = FourWayModel(params)
#for _ in range(params['steps']):
#    model_heur.step()
#print("Heurística:", model_heur.results())

#params_heur_r = params.copy()
#params_heur_r['ctrl_class'] = FourWaySignalsHeuristic

# (en la Parte 2 cambiaremos el spawn y el Car; por ahora, esto solo prueba el controlador)
#model_test = FourWayModel(params_heur_r)

#model_rotonda = FourWayModel(params_rotonda)
#for _ in range(params_rotonda['steps']):
#    model_rotonda.step()

#print("Resultados heuríst#ica en rotonda:", model_rotonda.results())


## **Función de animación (animation_plot)**

In [17]:
def draw_intersection(ax, L, w):
    ax.clear()
    ax.set_xlim(-L, L); ax.set_ylim(-L, L)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"Intersección | t = {model.t}s")

    # Calzada: dos carriles por sentido (solo dibujamos los centrales)
    # Horizontal
    ax.add_patch(plt.Rectangle((-L, -w), 2*L, 2*w, color='#e0e0e0', zorder=0))
    # Vertical
    ax.add_patch(plt.Rectangle((-w, -L), 2*w, 2*L, color='#e0e0e0', zorder=0))

    # Líneas centrales (guías)
    ax.plot([-L, L], [0, 0], color='white', lw=1, ls='--', zorder=1)
    ax.plot([0, 0], [-L, L], color='white', lw=1, ls='--', zorder=1)

    # Líneas de alto (2 m del centro)
    for x,y,dx,dy in [(-2, +w/2, 0, 1), (+2, -w/2, 0,-1), (-w/2, -2, -1,0), (+w/2, +2, 1,0)]:
        if dx==0: ax.plot([x-6, x+6],[y,y], color='yellow', lw=2, zorder=1)
        else:     ax.plot([x,x],[y-6,y+6], color='yellow', lw=2, zorder=1)


def my_plot(m, ax):
    L, w = m.p.L, m.p.w
    draw_intersection(ax, L, w)

    # Semáforos: círculos en esquinas del cruce
    lights = m.ctrl.lights()
    color_map = {'R':'#d32f2f','Y':'#f9a825','G':'#388e3c', 'AR':'#000000'} # Added 'AR':'#000000'
    # Ubicaciones aproximadas de focos por aproximación
    locs = {'N':(-w/2, +w), 'S':(+w/2, -w), 'E':(+w, +w/2), 'W':(-w, -w/2)}
    for d,(x,y) in locs.items():
        ax.add_patch(plt.Circle((x,y), 1.2, color=color_map[lights[d]], zorder=3))

    # Autos
    if len(m.cars) > 0:
        xs = [c.pos[0] for c in m.cars]
        ys = [c.pos[1] for c in m.cars]
        cs = ['#1976d2' if c.state!='stop' else '#455a64' for c in m.cars]
        ax.scatter(xs, ys, s=40, c=cs, edgecolor='k', linewidth=0.5, zorder=4)

## **Correr animación**

In [18]:
fig, ax = plt.subplots(figsize=(6,6))
model_rotonda = FourWayModel(params)
anim = ap.animate(model_rotonda , fig, ax, my_plot)
from IPython.display import HTML
HTML(anim.to_jshtml())


NameError: name 'model' is not defined